# 14. YOLO 출력 해석과 추론

이 노트북은 `13_YOLO_핵심_아이디어.ipynb`에서 본 grid, objectness, class score를 실제 추론 후처리 흐름으로 연결합니다.

YOLO 모델의 raw output은 사람이 바로 읽기 좋은 탐지 결과가 아닙니다. 모델 출력은 보통 많은 후보 박스를 담고 있고, 이 후보들을 좌표로 복원한 뒤 score threshold와 NMS를 거쳐 최종 결과로 정리합니다.

이번 노트북의 목표는 다음과 같습니다.

- YOLO 출력 텐서가 어떤 축으로 구성되는지 이해합니다.
- raw box 값을 이미지 좌표의 bounding box로 변환합니다.
- objectness와 class probability로 최종 class score를 계산합니다.
- score threshold와 class-wise NMS로 최종 탐지 결과를 만듭니다.
- 다음 노트북의 실제 이미지 추론 결과를 읽을 준비를 합니다.

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['axes.unicode_minus'] = False

## 14-1. YOLO 후처리 전체 흐름

YOLO 계열 모델의 세부 출력 형식은 버전마다 다릅니다. 하지만 사람이 최종적으로 보는 결과는 거의 같은 흐름으로 만들어집니다.

1. raw output을 후보 박스와 score로 해석합니다.
2. 박스 좌표를 원본 이미지 좌표계로 변환합니다.
3. 낮은 score의 후보를 제거합니다.
4. 같은 클래스 안에서 NMS를 적용합니다.
5. 남은 박스를 class, confidence, bbox로 표시합니다.

이 노트북에서는 작은 예제 텐서를 직접 만들어 흐름을 확인합니다.

## 14-2. 예제 grid와 class 준비

아래 예시는 `240 x 180` 이미지를 `4 x 3` grid로 나누고, 각 cell이 하나의 박스를 예측한다고 단순화합니다. 실제 YOLO는 여러 scale과 여러 후보 박스를 사용하지만, 후처리의 핵심은 같습니다.

In [ ]:
image_width, image_height = 240, 180
grid_cols, grid_rows = 4, 3
cell_w = image_width / grid_cols
cell_h = image_height / grid_rows
classes = ['cat', 'dog', 'car']

print('image size:', (image_width, image_height))
print('grid:', (grid_rows, grid_cols))
print('cell size:', (cell_w, cell_h))
print('classes:', classes)

## 14-3. raw prediction 만들기

하나의 예측은 다음 값을 가진다고 가정합니다.

- `cell`: 어떤 grid cell의 출력인지
- `tx`, `ty`: cell 내부에서의 중심점 상대 위치
- `tw`, `th`: 이미지 크기에 대한 박스 너비와 높이 비율
- `objectness`: 박스 안에 객체가 있을 가능성
- `class_probs`: 클래스별 확률

여기서는 sigmoid나 anchor decoding 같은 세부 구현은 생략하고, 이미 해석 가능한 값으로 만들어 둡니다.

In [ ]:
raw_predictions = [
    {'cell': (0, 1), 'tx': 0.42, 'ty': 0.58, 'tw': 0.34, 'th': 0.45, 'objectness': 0.92, 'class_probs': [0.84, 0.10, 0.06]},
    {'cell': (0, 1), 'tx': 0.48, 'ty': 0.62, 'tw': 0.32, 'th': 0.43, 'objectness': 0.78, 'class_probs': [0.80, 0.15, 0.05]},
    {'cell': (1, 3), 'tx': 0.10, 'ty': 0.85, 'tw': 0.30, 'th': 0.46, 'objectness': 0.88, 'class_probs': [0.09, 0.86, 0.05]},
    {'cell': (2, 0), 'tx': 0.60, 'ty': 0.45, 'tw': 0.22, 'th': 0.24, 'objectness': 0.35, 'class_probs': [0.12, 0.10, 0.78]},
    {'cell': (1, 2), 'tx': 0.45, 'ty': 0.45, 'tw': 0.20, 'th': 0.25, 'objectness': 0.20, 'class_probs': [0.20, 0.18, 0.62]},
]

for pred in raw_predictions:
    print(pred)

## 14-4. Cell 상대 좌표를 이미지 좌표로 변환하기

YOLO 출력은 보통 중심점 기반 좌표로 해석됩니다. 시각화나 IoU 계산에는 `x1, y1, x2, y2` 형식이 편하므로 변환 함수를 만듭니다.

In [ ]:
def decode_prediction(pred):
    row, col = pred['cell']
    cx = (col + pred['tx']) * cell_w
    cy = (row + pred['ty']) * cell_h
    w = pred['tw'] * image_width
    h = pred['th'] * image_height

    x1 = max(0, cx - w / 2)
    y1 = max(0, cy - h / 2)
    x2 = min(image_width, cx + w / 2)
    y2 = min(image_height, cy + h / 2)
    return (x1, y1, x2, y2)


for pred in raw_predictions:
    print(pred['cell'], '->', tuple(round(v, 1) for v in decode_prediction(pred)))

## 14-5. Objectness와 class probability 결합하기

탐지 결과의 confidence는 보통 `objectness x class probability`로 해석할 수 있습니다. 가장 점수가 높은 클래스를 대표 클래스로 선택합니다.

In [ ]:
def to_candidate(pred):
    box = decode_prediction(pred)
    scores = [pred['objectness'] * prob for prob in pred['class_probs']]
    class_id = max(range(len(scores)), key=lambda idx: scores[idx])
    return {
        'box': box,
        'class_id': class_id,
        'class': classes[class_id],
        'score': scores[class_id],
        'objectness': pred['objectness'],
        'class_probs': pred['class_probs'],
    }


candidates = [to_candidate(pred) for pred in raw_predictions]
for cand in candidates:
    print(f"{cand['class']:<3} score={cand['score']:.3f} box={tuple(round(v, 1) for v in cand['box'])}")

## 14-6. Score threshold 적용하기

score threshold는 낮은 확신의 후보를 먼저 제거합니다. threshold를 높이면 결과가 깔끔해지지만, 실제 객체를 놓칠 수 있습니다.

In [ ]:
score_threshold = 0.5
filtered = [cand for cand in candidates if cand['score'] >= score_threshold]

print('score threshold:', score_threshold)
for cand in filtered:
    print(f"{cand['class']:<3} score={cand['score']:.3f}")

## 14-7. NMS 함수 준비

NMS는 같은 클래스를 예측한 박스끼리 적용하는 것이 일반적입니다. 고양이 박스가 강아지 박스를 지우면 안 되기 때문입니다.

In [ ]:
def box_area_xyxy(box):
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)


def iou_xyxy(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_area = box_area_xyxy((inter_x1, inter_y1, inter_x2, inter_y2))

    union_area = box_area_xyxy(box_a) + box_area_xyxy(box_b) - inter_area
    return inter_area / union_area if union_area > 0 else 0.0


def nms_classwise(candidates, iou_threshold=0.5):
    final = []
    for class_id in sorted({cand['class_id'] for cand in candidates}):
        same_class = [cand for cand in candidates if cand['class_id'] == class_id]
        same_class = sorted(same_class, key=lambda cand: cand['score'], reverse=True)

        while same_class:
            best = same_class.pop(0)
            final.append(best)
            same_class = [
                cand for cand in same_class
                if iou_xyxy(best['box'], cand['box']) < iou_threshold
            ]

    return sorted(final, key=lambda cand: cand['score'], reverse=True)


final_detections = nms_classwise(filtered, iou_threshold=0.5)
for det in final_detections:
    print(f"{det['class']:<3} score={det['score']:.3f} box={tuple(round(v, 1) for v in det['box'])}")

## 14-8. 후처리 결과 시각화

이제 후보 박스와 최종 박스를 나란히 비교합니다. 중복된 고양이 후보가 NMS 후 하나로 정리되는지 확인합니다.

In [ ]:
def draw_detections(ax, detections, title):
    color_map = {'cat': 'crimson', 'dog': 'royalblue', 'car': 'seagreen'}
    ax.set_xlim(0, image_width)
    ax.set_ylim(image_height, 0)
    ax.set_facecolor('#f8fafc')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

    for det in detections:
        x1, y1, x2, y2 = det['box']
        color = color_map.get(det['class'], 'black')
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2.5))
        ax.text(x1, max(10, y1 - 4), f"{det['class']} {det['score']:.2f}", color=color, fontsize=10, weight='bold')


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
draw_detections(axes[0], filtered, 'Score threshold 후 후보')
draw_detections(axes[1], final_detections, 'Class-wise NMS 후 최종 결과')
plt.tight_layout()
plt.show()

## 14-9. threshold를 바꾸며 결과 관찰하기

실제 추론에서는 `score_threshold`와 `iou_threshold`를 조절하면서 결과를 확인합니다.

- score threshold가 낮으면 더 많은 후보가 남습니다.
- IoU threshold가 낮으면 중복 제거가 강해집니다.
- IoU threshold가 높으면 겹치는 박스도 더 많이 남습니다.

In [ ]:
for score_th in [0.2, 0.5, 0.7]:
    for iou_th in [0.3, 0.5, 0.8]:
        filtered_now = [cand for cand in candidates if cand['score'] >= score_th]
        final_now = nms_classwise(filtered_now, iou_threshold=iou_th)
        labels = [f"{det['class']}:{det['score']:.2f}" for det in final_now]
        print(f"score_th={score_th}, iou_th={iou_th} -> {labels}")

## 정리

- YOLO 출력은 많은 후보 박스를 담고 있으므로 바로 최종 결과가 아닙니다.
- cell 상대 좌표와 박스 크기를 이미지 좌표의 `xyxy` 박스로 변환해야 합니다.
- `objectness x class probability`를 통해 class별 final score를 계산할 수 있습니다.
- score threshold로 낮은 확신의 후보를 제거하고, class-wise NMS로 중복 박스를 정리합니다.
- 실제 YOLO 라이브러리가 반환하는 결과도 결국 `class`, `confidence`, `bbox` 중심으로 읽으면 됩니다.

다음 노트북 `15_YOLO_이미지_실습.ipynb`에서는 실제 이미지 또는 예제 이미지에 YOLO 추론을 적용하고 결과를 시각화합니다.